In [1]:
library(CMAP) 
library(Seurat) 
library(e1071)
library(purrr)  
library(dplyr)
library(preprocessCore)
library(reticulate)
library(smfishHmrf)
library(Giotto)
#use_python("/home/qyyuan/anaconda3/envs/CMAP/bin/python", required = TRUE)
python_path <- '/home/qyyuan/anaconda3/envs/CMAP/bin/python'
use_condaenv(python_path)
save_directory <- "/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/CMAP"
if(!file.exists(save_directory)) dir.create(save_directory, recursive = T)

Attaching SeuratObject

Seurat v4 was just loaded with SeuratObject v5; disabling v5 assays and
validation routines, and ensuring assays work in strict v3/v4
compatibility mode


载入程辑包：‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


载入需要的程辑包：pracma


载入程辑包：‘pracma’


The following object is masked from ‘package:purrr’:

    cross


The following object is masked from ‘package:e1071’:

    sigmoid


载入需要的程辑包：fs



In [2]:
sc_counts <- read.csv("/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/spaTrio/multi_rna.csv",row.names = 1, check.names=FALSE)
spatial_count <- read.csv("/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/spaTrio/spatial_rna.csv",row.names = 1, check.names=FALSE)
sc_meta <- read.csv(
    "/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/spaTrio/multi_meta.csv",
    check.names = FALSE,
    colClasses = "character"   # 关键：全部按字符读取
)
spatial_location <- read.csv("/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/spaTrio/pos.csv",row.names = 1, check.names=FALSE)
sc_counts <- sc_counts[rowSums(sc_counts)>0,]
sc_norm = as.matrix(log1p(sweep(sc_counts,2,Matrix::colSums(sc_counts),FUN = '/') * 1e4))

spatial_count <- spatial_count[rowSums(spatial_count)>0,]
st_norm = log1p(sweep(spatial_count,2,Matrix::colSums(spatial_count),FUN = '/') * 1e4)
rownames(sc_meta)= sc_meta$`Cell IDs`
spatial_count <- as.matrix(spatial_count)
sc_counts=as.matrix(sc_counts)

In [3]:
df=spatial_location
df$x <- (df$x - min(df$x, na.rm = TRUE)) / (max(df$x, na.rm = TRUE) - min(df$x, na.rm = TRUE))
df$y <- (df$y - min(df$y, na.rm = TRUE)) / (max(df$y, na.rm = TRUE) - min(df$y, na.rm = TRUE))
spatial_location=df
spatial_location$cell_ID =rownames(spatial_location)

In [4]:
cluster_k <- 3
# Create specific instructions for Giotto analysis workflow
instrs <- createGiottoInstructions(save_plot = TRUE,
                                   show_plot = TRUE,
                                   return_plot = TRUE,
                                   python_path = python_path,
                                   save_dir = save_directory)
spatial_obj <- createGiottoObject(raw_exprs = spatial_count,
                                  spatial_locs = spatial_location[,c('x','y')],
                                  instructions = instrs,
                                  cell_metadata = spatial_location)
# Filter genes and cells. If you have filtered some low quality spots before, you can skip this step
# spatial_obj <- filterGiotto(gobject = spatial_obj,
#                             expression_threshold = 1,
#                             gene_det_in_min_cells = 1,
#                             min_det_genes_per_cell = 1,
#                             expression_values = c('raw'),
#                             verbose = T)





 external python path provided and will be used 
Consider to install these (optional) packages to run all possible Giotto commands for spatial analyses:  scran MAST trendsceek SPARK multinet RTriangle FactoMiner
 Giotto does not automatically install all these packages as they are not absolutely required and this reduces the number of dependencies

In [5]:
spatial_obj <- normalizeGiotto(gobject = spatial_obj, scalefactor = 60, verbose = T)


 first scale genes and then cells 


In [6]:
# Create spatial network
#@ maximum_distance_knn: Visium data, tissue_hires_scalef set ceiling(24.8/tissue_hires_scalef) or as maximum_distance_knn,  tissue_hires_scalef is saved in scalefactors_json.json; slide-seq/ST: set 1.5
spatial_obj <- createSpatialNetwork(gobject = spatial_obj,
                                    method = 'kNN',
                                    k = 6, # this k represents the number of neighbors
                                    maximum_distance_knn = 370, 
                                    minimum_k = 1,
                                    name = 'KNN_network')


In [7]:
kmtest  <- binSpect(spatial_obj, calc_hub = T, hub_min_int = 5,spatial_network_name = 'KNN_network')

hmrf_folder = paste0(save_directory,'/11_HMRF')
if(!file.exists(hmrf_folder)) dir.create(hmrf_folder, recursive = T)
spatial_genes_selected <- hmrf_spatial_gene(spatial_obj,
                                            kmtest,
                                            k = cluster_k) # k: Number of spatial domains; set according to your data.

#@ betas: For detailed settings, see https://search.r-project.org/CRAN/refmans/smfishHmrf/html/smfishHmrf.hmrfem.multi.it.min.html
# For quick results, we recomoned setting betas to 45(non-tumor) or 0(tumor sample).
# If you don't mind taking more time and want the best results, you can iteratively test values between 0 and 100 and select the best one.
HMRF_spatial_genes = doHMRF(gobject = spatial_obj,
                            expression_values = 'scaled',
                            spatial_genes = spatial_genes_selected,
                            k = cluster_k, # This value should match the number of spatial domains (k).
                            spatial_network_name="KNN_network",
                            betas = c(0, 45, 2), 
                            python_path = python_path,
                            output_folder = paste0(hmrf_folder, '/', 'Spatial_genes/SG_topgenes_elbow_k_scaled'))



 This is the single parameter version of binSpect
 1. matrix binarization complete 

 2. spatial enrichment test completed 

 3. (optional) average expression of high expressing cells calculated 

 4. (optional) number of high expressing cells calculated 

Elbow method chosen to determine number of spatial genes.

Elbow point determined to be at x=15 genes y=1.85206967846308

 Removed 239 from user's input gene list due to being absent or non-spatial genes.

 Kept 15 spatial genes for the sampling step next

 Will use 15 genes for init of HMRF.

 expression_matrix.txt already exists at this location, will be overwritten 

 spatial_genes.txt already exists at this location, will be overwritten 

 spatial_network.txt already exists at this location, will be overwritten 

 spatial_cell_locations.txt already exists at this location, will be overwritten 
[1] "/home/qyyuan/anaconda3/envs/CMAP/bin/python /data/qyyuan/anaconda3/envs/CMAP/lib/R/library/Giotto/python/reader2.py -l \"/home/qyyua

In [8]:
#@betas_to_add: Results from different betas that you want to add
# Recommendations: Tumor sample: beta=0; Non-tumor: beta=45.
beta = 0
spatial_obj = addHMRF(gobject = spatial_obj,
                      HMRFoutput = HMRF_spatial_genes,
                      k = cluster_k,
                      betas_to_add = beta,  # according to the above beta settings
                      hmrf_name = 'HMRF')
# Add spatial domain to spatial metadata. You can also save the spatial_location as an intermediate file, which must include spatial genes and spatial cluster labels.


[1] "/home/qyyuan/anaconda3/envs/CMAP/bin/python /data/qyyuan/anaconda3/envs/CMAP/lib/R/library/Giotto/python/get_result2.py -r \"/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/CMAP/11_HMRF/Spatial_genes/SG_topgenes_elbow_k_scaled/result.spatial.zscore\" -a test -k 3 -b 0"


In [9]:
spatial_location = spatial_location[as.data.frame(spatial_obj@cell_metadata)[,'cell_ID'],]
column <- paste0('HMRF_k',cluster_k,'_b.',beta)
spatial_location = cbind(spatial_location,HMRF_cluster = spatial_obj@cell_metadata[,..column]) # this coloumn needs to be set as described above (the number of domains and beta)
st_norm = st_norm[,rownames(spatial_location)]

In [10]:
matrix <- data_to_transform(sc_norm,st_norm,spatial_genes_selected,batch=TRUE,pca_method='prcomp_irlba',npc = 10)


Warning message in (function (A, nv = 5, nu = nv, maxit = 1000, work = nv + 7, reorth = TRUE, :
“You're computing too large a percentage of total singular values, use a standard svd instead.”


In [11]:
train_set <- cbind(as.data.frame(t(matrix[,colnames(st_norm)])),label=spatial_location$`HMRF_k3_b.0`)


In [12]:
test_set <- as.data.frame(t(matrix[,colnames(sc_norm)]))


In [13]:
train_set$label = as.factor(train_set$label)
# Predict spatial domain of individual cells
# This tuning step requires some time. You can adjust the cross-validation proportion using `cross_para` parameter in the `tune_parameter()` function.


In [14]:
tune_parameter1=function (train_set, test_set, scale = TRUE, class.weight = TRUE, 
    kernel = "radial", verbose = FALSE, cross_para = c(4, 6, 
        8, 10)) 
{
    parameter <- list()
    if (scale) {
        scale_data = scale_data_by_column(train_set, test_set)
        tmp.train = scale_data[[1]]
        tmp.test = scale_data[[2]]
    }
    else {
        print("Please set Scale TRUE")
        tmp.train = train_set
        tmp.test = test_set
    }
    if (class.weight) {
        num_cluster <- length(unique(tmp.train$label))
        wts = dim(tmp.train)[1]/table(tmp.train$label)/length(unique(tmp.train$label))
    }
    else {
        wts = rep(1, length(unique(tmp.train$label)))
        names(wts) <- names(table(tmp.train$label))
    }
    for (cross_i in cross_para) {
        set.seed(321)
        tune.out.corse = e1071::tune(e1071::svm, label ~ ., data = tmp.train, 
            kernel = kernel, ranges = list(cost = 2^seq(-5, 15, 
                2), gamma = 2^seq(-15, 3, 2)), probability = TRUE, 
            class.weight = wts, scale = FALSE, tunecontrol = e1071::tune.control(cross = cross_i))
        bestmodel <- tune.out.corse$best.model
        parameter[[paste0("cross_", cross_i)]][["cost"]] <- bestmodel$cost
        parameter[[paste0("cross_", cross_i)]][["gamma"]] <- bestmodel$gamma
        if (verbose) {
            print(paste0("SVM: cross=", cross_i))
            print("best parameters:")
            print(paste0("cost:", bestmodel$cost))
            print(paste0("gamma:", bestmodel$gamma))
        }
    }
    return(parameter)
}

In [15]:
parameters <- tune_parameter(train_set, test_set, kernel = "radial", scale = TRUE, class.weight = TRUE, verbose = TRUE, cross_para=4)


[1] "SVM: cross=4"
[1] "best parameters:"
[1] "cost:8"
[1] "gamma:2"


In [16]:
pred_st_svm <- PredictDomain(train_set, test_set, cost=parameters[['cross_4']][['cost']],
                             gamma=parameters[['cross_4']][['gamma']], st_svm=TRUE,verbose = FALSE)
pred_sc_svm <- PredictDomain(train_set, test_set, cost=parameters[['cross_4']][['cost']],
                             gamma=parameters[['cross_4']][['gamma']], scale = TRUE, verbose = TRUE)


[1] "The prediction table of training data"
pred_st_svm
  1   2   3 
146 158 120 
[1] "The accuracy of spatial data:0.89622641509434"
[1] "The prediction table of test data"
pred_sc_svm
   1    2    3 
1377 1482 1339 


In [17]:
sc_meta <- sc_meta[apply(attr(pred_sc_svm, "probabilities"),1,max)>0.8,] 
pred_sc_svm <- pred_sc_svm[apply(attr(pred_sc_svm, "probabilities"),1,max)>0.8]
sc_norm <- sc_norm[,rownames(sc_meta)]

In [18]:
map_spot_genes_new=function (sc_norm, sc_meta, st_norm, spatial_location) 
{
    st <- Seurat::CreateSeuratObject(counts = st_norm, meta.data = spatial_location, 
        min.cells = 0, min.features = 0, assay = "Spatial")
    st@assays$Spatial@data <- as.matrix(st_norm)
    Idents(st) <- st@meta.data$'HMRF_k3_b.0'
    cluster <- sort(unique(st@meta.data$'HMRF_k3_b.0'))
    st_gene_list <- list()
    for (i in 1:length(cluster)) {
        sub_st <- subset(st, idents = cluster[i])
        sub_st <- sub_st[!apply(sub_st@assays$Spatial@data, 1, 
            var) == 0, ]
        sub_st <- Seurat::FindVariableFeatures(sub_st, selection.method = "vst", 
            nfeatures = 3000)
        st_gene_list[[i]] <- intersect(rownames(sc_norm), sub_st@assays$Spatial@var.features)
    }
    return(st_gene_list)
}

In [19]:
assignInNamespace("map_spot_genes", map_spot_genes_new, ns = "CMAP")
cell_spot_map <- map_cell_to_spot(sc_norm=sc_norm,sc_meta=sc_meta,
                                  st_norm=st_norm,spatial_location=spatial_location,
                                  pred_sc_svm=pred_sc_svm, pred_st_svm=pred_st_svm,
                                  python_path=python_path,
                                  batch=TRUE,
                                  num_epochs=2000L,
                                  para_distance=1.0,
                                  para_density=1.0)

Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.
ℹ The deprecated feature was likely used in the Seurat package.
  Please report the issue at <https://github.com/satijalab/seurat/issues>.”


In [23]:
library(tidyr)
library(dplyr)

mat_df <- cell_spot_map %>%
  pivot_wider(
    id_cols = Single_cell,
    names_from = Spot,
    values_from = Probability,
    values_fill = 0        # 缺失的组合填 0
  )

# 转成矩阵（第一列是行名）
mat <- as.matrix(mat_df[, -1])
rownames(mat) <- mat_df$Single_cell

In [25]:
write.table(mat,file = '/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/CMAP/cell_to_spot.txt',quote = FALSE)

In [20]:
spot_neigh_list <- spatial_relation_all(spatial_location,
                                        spatial_data_type=c('honeycomb'))

sc_meta_coord <- calculate_cell_location(cell_spot_map=cell_spot_map,
                                         st_meta =spatial_location,
                                         sc_meta=sc_meta,
                                         sc_norm=sc_norm,
                                         st_norm=st_norm,
                                         parallel = TRUE,                   
                                         batch = TRUE,
                                         spot_neigh_list=spot_neigh_list,
                                         radius = 1)

In [28]:
sc_meta_coord
write.table(sc_meta_coord,file = '/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/CMAP/coodinate_spot.txt',quote = FALSE)

,Cell IDs,id,type,pred_loc_x,pred_loc_y
,<chr>,<chr>,<chr>,<dbl>,<dbl>
100416591302420540942373170232872783233,100416591302420540942373170232872783233,100416591302420540942373170232872783233,L4_5_IT,0.88100730,0.74074074
100542677892296141453580613270127987555,100542677892296141453580613270127987555,100542677892296141453580613270127987555,L4_5_IT,0.96016580,0.51851852
102500510899266198503869053721430715211,102500510899266198503869053721430715211,102500510899266198503869053721430715211,L6_IT,0.23608496,0.37037037
103384017201440493716854958896156052945,103384017201440493716854958896156052945,103384017201440493716854958896156052945,L6_IT,0.56910987,0.18518519
104078254148375929483440785584365448099,104078254148375929483440785584365448099,104078254148375929483440785584365448099,L4_5_IT,0.33334536,0.18518519
10551633534698854195667512244395380368,10551633534698854195667512244395380368,10551633534698854195667512244395380368,L5_IT,0.44128261,0.14814815
106099584344502602441591320617159624141,106099584344502602441591320617159624141,106099584344502602441591320617159624141,L4_5_IT,0.61869071,0.07407407
107624669841857750247431772682217706748,107624669841857750247431772682217706748,107624669841857750247431772682217706748,L6_IT,0.23030563,0.22222222
107811399342642680792197384207607904351,107811399342642680792197384207607904351,107811399342642680792197384207607904351,L2_3_IT,0.36136545,0.66666667
